In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostRegressor

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s4e5/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s4e5/test.csv")

In [ ]:
train.columns

In [ ]:
train.info()

In [ ]:
train.head()

In [ ]:
train.isnull().sum()

In [ ]:
train.drop('id',axis=1,inplace=True)


In [ ]:
train.columns

In [ ]:
train.describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(train["FloodProbability"], bins=30, kde=True)
plt.title("Flood Probability Distribution")
plt.show()

In [ ]:
corr = train.corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr,
            cmap="coolwarm",
            center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
target_corr = corr["FloodProbability"].sort_values(ascending=False)

print(target_corr)

In [ ]:
x_temp = train.drop(['FloodProbability'],axis=1)
y_temp = train['FloodProbability']

model = CatBoostRegressor(
    iterations = 300,
    verbose=0
)

model.fit(x_temp,y_temp)


In [ ]:
importance = pd.DataFrame({
    "Feature":x_temp.columns,
    "Importance":model.feature_importances_
})
importance = importance.sort_values(
    by="Importance",
    ascending=False
)
plt.figure(figsize=(10,8))
sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

In [ ]:
print(importance.head())
print(importance.columns)

In [ ]:
features = train.drop(
    ["FloodProbability"],
    axis=1
)

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

In [ ]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

train["Cluster"] = kmeans.fit_predict(
    scaled_features
)

train["Cluster"].value_counts()

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

pca_data = pca.fit_transform(
    scaled_features
)

plt.figure(figsize=(10,6))

scatter = plt.scatter(
    pca_data[:,0],
    pca_data[:,1],
    c=train["Cluster"],
    cmap="viridis"
)

plt.colorbar(scatter)
plt.title("KMeans Clusters")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x=train["Cluster"],
    y=train["FloodProbability"]
)

plt.show()

In [ ]:
risk_features = [
    'MonsoonIntensity',
    'Deforestation',
    'Urbanization',
    'ClimateChange',
    'Siltation',
    'CoastalVulnerability'
]

train["RiskScore"] = train[risk_features].mean(axis=1)
test["RiskScore"] = test[risk_features].mean(axis=1)

In [ ]:
infra = [
    'DamsQuality',
    'DrainageSystems',
    'RiverManagement'
]

train["InfraScore"] = train[infra].mean(axis=1)
test["InfraScore"] = test[infra].mean(axis=1)

In [ ]:
env = [
    'Watersheds',
    'WetlandLoss',
    'Landslides'
]

train["EnvScore"] = train[env].mean(axis=1)
test["EnvScore"] = test[env].mean(axis=1)

In [ ]:
train["Monsoon_Urban"] = (
    train["MonsoonIntensity"] *
    train["Urbanization"]
)

test["Monsoon_Urban"] = (
    test["MonsoonIntensity"] *
    test["Urbanization"]
)

In [ ]:
train["Climate_Deforestation"] = (
    train["ClimateChange"] *
    train["Deforestation"]
)

test["Climate_Deforestation"] = (
    test["ClimateChange"] *
    test["Deforestation"]
)

In [ ]:
X = train.drop(
    ["FloodProbability"],
    axis=1
)

y = train["FloodProbability"]

X_test = test.drop(
    ["id"],
    axis=1
)

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=200
)

In [ ]:
oof = np.zeros(len(X))

test_preds = np.zeros(len(X_test))

for train_idx, valid_idx in kf.split(X):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid,y_valid),
        use_best_model=True
    )

    preds = model.predict(X_valid)

    oof[valid_idx] = preds

    test_preds += (
        model.predict(X_test)
        / kf.n_splits
    )

score = r2_score(y,oof)

print("CV R2 =",score)

In [ ]:
print(X.columns.tolist())
print(X_test.columns.tolist())